# Debug Mode

Start and inspect the Tango control stack in database mode before running acquisition notebooks.

Make sure you are on the VPN, the AutoScript server is running, and the Tango servers have been started with:

```bash
uv run scripts/run_servers.py
```


### Alternative local launch


In [ ]:
import sys
from pathlib import Path
import time
import json

# For resolving ModuleNotFoundErrors
notebook_dir = Path.cwd()
parent_dir = notebook_dir.parent.resolve()
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

import subprocess
import numpy as np
import matplotlib.pyplot as plt
from asyncroscopy.ThermoMicroscope import ThermoMicroscope
from tango.test_context import MultiDeviceTestContext
from asyncroscopy.detectors.EDS import EDS
from asyncroscopy.hardware.SCAN import SCAN
from asyncroscopy.software.DATA import DATA

import tango

# ── Kill anything already on port 11000 ──────────────────────────────────────
print("Clearing old processes...")
subprocess.run("kill -9 $(lsof -t -i:11000) 2>/dev/null || true", shell=True)
subprocess.run("pkill -f 'STAGE stage' 2>/dev/null || true", shell=True)
subprocess.run("pkill -f 'SCAN scan' 2>/dev/null || true", shell=True)
subprocess.run("pkill -f 'ThermoMicroscope microscope_instance' 2>/dev/null || true", shell=True)
time.sleep(2)


devices_info = [
    {
        "class": SCAN,
        "devices": [
            {
                "name": "asyncroscopy/scan/default",
                "properties": {},
            }
        ],
    },

    {
        "class": EDS,
        "devices": [
            {
                "name": "asyncroscopy/eds/default",
                "properties": {},
            }
        ],
    },
    

    {
        "class": DATA,
        "devices": [
            {
                "name": "asyncroscopy/data/default",
                "properties": {},
            }
        ],
    },
    {
        "class": ThermoMicroscope,
        "devices": [
            {
                "name": "asyncroscopy/microscope/default",
                "properties": {
                    "eds_device_address": "asyncroscopy/eds/default",
                    "scan_device_address": "asyncroscopy/scan/default",
                    "data_device_address": "asyncroscopy/data/default",
                },
            }
        ],
    },
]

ctx = MultiDeviceTestContext(devices_info, process=False)
ctx.start()

scan_proxy = tango.DeviceProxy("asyncroscopy/scan/default")
mic_proxy = tango.DeviceProxy("asyncroscopy/microscope/default")
eds_proxy = tango.DeviceProxy("asyncroscopy/eds/default")
data_proxy = tango.DeviceProxy("asyncroscopy/data/default")

print(f"Device state: {mic_proxy.state()}")


### Connect to devices


In [ ]:
import os

# Tango DB running on this
os.environ["TANGO_HOST"] = "localhost:9000"
# os.environ["TANGO_HOST"] = "10.46.217.241:9094"

In [ ]:
# list devices on DB
import tango

db = tango.Database()

devices = db.get_device_name("*", "*")

print("Devices registered in Tango DB:\n")

for d in devices:
    print(d)

### Interpret registered Tango devices


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import tango
from tiled.client import from_uri

microscope_proxy = tango.DeviceProxy("asyncroscopy/microscope/default")
microscope_proxy.set_timeout_millis(120_000)

scan_proxy = tango.DeviceProxy("asyncroscopy/scan/default")
data_proxy = tango.DeviceProxy("asyncroscopy/data/default")
data_proxy.set_timeout_millis(120_000)

TILED_HOST = "localhost"
TILED_PORT = 9091
save_path = "outputs/tiled_acquisitions"

data_proxy.host = TILED_HOST
data_proxy.port = TILED_PORT
data_proxy.save_path = save_path

if str(data_proxy.tiled_server).lower() != "yes":
    config = json.loads(data_proxy.start_tiled_server())
else:
    config = json.loads(data_proxy.get_config())

client = from_uri(config.get("uri", f"http://{TILED_HOST}:{TILED_PORT}"))
print(json.dumps(config, indent=2))


In [ ]:
print('Microscope state:', microscope_proxy.state())


### Inspect device attributes and commands


In [ ]:
print('\n--- Microscope commands ---')
for cmd in microscope_proxy.get_command_list():
    print(f'  {cmd}')

### Configure HAADF acquisition


In [ ]:
scan_proxy.dwell_time 

In [ ]:
scan_proxy.dwell_time   = 10e-6   # 1 µs
scan_proxy.imsize  = 1024


print('dwell_time  :', scan_proxy.dwell_time)
print('image_width :', scan_proxy.imsize)


### Acquire a HAADF image


In [ ]:
image_key = microscope_proxy.acquire_scanned_image()
image_node = client[image_key]
image = np.asarray(image_node.read())
metadata = dict(image_node.metadata)

print("Tiled key  :", image_key)
print("Metadata   :", metadata)
print("Image shape:", image.shape)
print("Image dtype:", image.dtype)


### Display the image


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image, cmap='gray', interpolation='none')
ax.set_title(f"HAADF — dwell {metadata['dwell_time']*1e6:.1f} µs")
ax.axis('off')
plt.tight_layout()
plt.show()

### Build a sidpy dataset


In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

import sidpy
print('sidpy version: ', sidpy.__version__)

In [ ]:
dataset = sidpy.Dataset.from_array(image , name='random')

# set dimesnions
dataset.set_dimension(0, sidpy.Dimension(np.arange(image.shape[0])*.02, 'x'))
dataset.set_dimension(1, sidpy.Dimension(np.arange(image.shape[0])*.02, 'y'))


# set the dataset level plotting metadata
dataset.data_type = 'image'
dataset.units = 'counts'
dataset.quantity = 'intensity'
dataset.title = 'random'

# handle one dimension of the data
dataset.set_dimension(0, sidpy.Dimension(np.arange(dataset.shape[0])*.02, 'x'))
dataset.x.dimension_type = 'spatial'
dataset.x.units = 'nm'
dataset.x.quantity = 'distance'

# handle another dimension of the data

dataset.set_dimension(1, sidpy.Dimension(np.arange(dataset.shape[1])*.02, 'y'))
dataset.y.dimension_type = 'spatial'
dataset.y.units = 'nm'
dataset.y.quantity = 'distance'


In [ ]:
dataset.metadata = metadata

In [ ]:
view = dataset.plot(scale_bar=True)

### Advanced multi-detector acquisition


In [ ]:
adv_acq_proxy = tango.DeviceProxy("tango://127.0.0.1:8890/asyncroscopy/advancedacquisition/default#dbase=no")


In [ ]:
adv_acq_proxy.state()

In [ ]:
for attr in adv_acq_proxy.get_attribute_list():
    print(f'  {attr}')



In [ ]:
adv_acq_proxy.dwell_time

In [ ]:
adv_acq_proxy.scan_region

In [ ]:
# One simultaneous acquisition
image_keys = json.loads(microscope_proxy.acquire_images(["HAADF", "BF"]))
image_keys


In [ ]:
import matplotlib.pyplot as plt

images = []
for key in image_keys:
    node = client[key]
    img = np.asarray(node.read())
    metadata = dict(node.metadata)
    images.append((metadata.get("detector", key), key, img, metadata))

fig, axes = plt.subplots(1, len(images), figsize=(6 * len(images), 5))
if len(images) == 1:
    axes = [axes]

for ax, (detector_name, key, img, metadata) in zip(axes, images):
    im = ax.imshow(img, cmap="gray")
    ax.set_title(str(detector_name).upper())
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()


### Convert acquired images to sidpy datasets


In [ ]:
import numpy as np
import sidpy

image_keys = json.loads(microscope_proxy.acquire_images(["HAADF", "BF"]))
datasets = []

for key in image_keys:
    node = client[key]
    img = np.asarray(node.read())
    metadata = dict(node.metadata)
    detector_name = metadata.get("detector", key)
    dataset = sidpy.Dataset.from_array(img, name=str(detector_name))
    dataset.set_dimension(0, sidpy.Dimension(np.arange(img.shape[0]) * 0.02, "x"))
    dataset.set_dimension(1, sidpy.Dimension(np.arange(img.shape[1]) * 0.02, "y"))
    dataset.data_type = "image"
    dataset.units = "counts"
    dataset.quantity = "intensity"
    dataset.title = str(detector_name).upper()
    dataset.metadata = metadata
    dataset.x.dimension_type = "spatial"
    dataset.x.units = "nm"
    dataset.x.quantity = "distance"
    dataset.y.dimension_type = "spatial"
    dataset.y.units = "nm"
    dataset.y.quantity = "distance"
    datasets.append(dataset)

datasets


In [ ]:
datasets